# 03 - Evaluate and predict

Use this notebook after training. It validates the trained single-class fracture model and writes prediction images under `runs/fracture/predict`.


In [ ]:
RUN_ENV = "local"  # "colab" or "kaggle"
PROJECT_NAME = "yolov8-fracture-detection"
IMAGE_SIZE = 640
CONFIDENCE = 0.25


In [ ]:
from pathlib import Path
if RUN_ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_NAME
elif RUN_ENV == "kaggle":
    PROJECT_ROOT = Path("/kaggle/working") / PROJECT_NAME
else:
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

DATA_YAML = PROJECT_ROOT / "data" / "fracture_subset" / "fracture.yaml"
WEIGHTS_PATH = PROJECT_ROOT / "weights" / "fracture_yolov8n_best.pt"
RUNS_ROOT = PROJECT_ROOT / "runs"


print(f"Dataset YAML: {DATA_YAML}")
print(f"Weights: {WEIGHTS_PATH}")


In [ ]:
if RUN_ENV in {"colab", "kaggle"}:
    %pip install -U ultralytics


In [ ]:
from ultralytics import YOLO


def validate_single_class_labels(dataset_root):
    root = Path(dataset_root).expanduser()
    bad_lines = []
    for label_path in sorted((root / "labels").glob("**/*.txt")):
        for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
            stripped = line.strip()
            if not stripped:
                continue
            parts = stripped.split()
            if len(parts) != 5 or parts[0] != "0":
                bad_lines.append(f"{label_path}:{line_number}: {stripped}")
    if bad_lines:
        preview = "\n".join(bad_lines[:10])
        raise ValueError(f"Labels must be YOLO detect rows with class id 0 only:\n{preview}")

if not DATA_YAML.exists():
    raise FileNotFoundError(f"Missing dataset YAML: {DATA_YAML}")
if not WEIGHTS_PATH.exists():
    raise FileNotFoundError(f"Missing trained weights: {WEIGHTS_PATH}")
validate_single_class_labels(DATA_YAML.parent)

model = YOLO(str(WEIGHTS_PATH))
metrics = model.val(data=str(DATA_YAML), imgsz=IMAGE_SIZE, project=str(RUNS_ROOT / "fracture"), name="val", exist_ok=True)
metrics


In [ ]:
# Predict on the prepared test images. Change SOURCE_IMAGES to any fracture X-ray folder you want to inspect.
SOURCE_IMAGES = DATA_YAML.parent / "images" / "test"

prediction_results = model.predict(
    source=str(SOURCE_IMAGES),
    imgsz=IMAGE_SIZE,
    conf=CONFIDENCE,
    save=True,
    project=str(RUNS_ROOT / "fracture"),
    name="predict",
    exist_ok=True,
)
print(f"Prediction images saved under: {RUNS_ROOT / 'fracture' / 'predict'}")


In [ ]:
# Display a few saved predictions in the notebook.
from IPython.display import Image, display

predict_dir = RUNS_ROOT / "fracture" / "predict"
for image_path in sorted(predict_dir.glob("*.jpg"))[:6]:
    display(Image(filename=str(image_path)))
